In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# =================================================================
# 1. CARGA Y CORRUPCIÓN CONTROLADA (Paso 3.1 del documento)
# =================================================================
# Cargamos el dataset de Credit Card Fraud Detection
df = pd.read_csv('creditcard.csv')

# Definición de variables según el requerimiento:
# Variable Categórica Objetivo: 'Class' (0: Normal, 1: Fraude)
# Variable Numérica Continua: 'Amount' (Monto de la transacción)
cat_var = 'Class'
col_num = 'Amount'

# Semilla basada en los últimos 3 dígitos de la boleta
mi_boleta = 123  # <--- REEMPLAZA CON TUS DATOS REALES
np.random.seed(mi_boleta)

# Inyección artificial del 15% de valores nulos (NaN)
indices_nulos = np.random.choice(df.index, size=int(len(df) * 0.15), replace=False)
df.loc[indices_nulos, col_num] = np.nan

print(f"--- Fase 1: Corrupción ---")
print(f"Nulos inyectados en {col_num}: {df[col_num].isna().sum()}\n")

# =================================================================
# 2. IMPUTACIÓN CONTEXTUAL (Paso 3.2 del documento)
# =================================================================
# Restauramos la calidad usando la mediana específica de cada categoría
# Evitamos el sesgo global aplicando lógica local por estrato
df[col_num] = df.groupby(cat_var)[col_num].transform(lambda x: x.fillna(x.median()))

# =================================================================
# 3. DETECCIÓN DE OUTLIERS LOCALES (Z-SCORE) (Paso 3.3 del documento)
# =================================================================
# Implementación matemática del Z-Score agrupado por variable categórica
# Fórmula: (x - media_grupo) / desviacion_grupo
z_scores = df.groupby(cat_var)[col_num].transform(lambda x: (x - x.mean()) / x.std())

# Filtrado y eliminación de registros donde |Z| > 3 [cite: 26]
df_limpio = df[z_scores.abs() <= 3].copy()

print(f"--- Fase 3: Limpieza de Outliers ---")
print(f"Registros eliminados por Z-Score local: {len(df) - len(df_limpio)}\n")

# =================================================================
# 4. MUESTREO ESTRATIFICADO (Paso 3.4 del documento)
# =================================================================
# Extraemos el 20% manteniendo la proporción original de las clases
# Aquí implementamos el 'random_state' con la semilla de la boleta
df_muestra, _ = train_test_split(
    df_limpio,
    train_size=0.20,
    stratify=df_limpio[cat_var],
    random_state=mi_boleta  # <--- IMPLEMENTACIÓN DE SEMILLA REQUERIDA
)

# =================================================================
# 5. SUAVIZADO DE DATOS (Paso 3.5 del documento)
# =================================================================
# Aplicamos Binning de igual frecuencia con 4 contenedores
df_muestra['Bin'] = pd.qcut(df_muestra[col_num], q=4, labels=False)

# Reemplazo de valores por la media de su propio contenedor
df_muestra[col_num] = df_muestra.groupby('Bin')[col_num].transform('mean')

# =================================================================
# RESULTADOS PARA EL REPORTE
# =================================================================
print(f"--- Fase final: Muestreo y Suavizado ---")
print(f"Tamaño de la muestra final: {len(df_muestra)}")
print("\nDistribución de clases (Estratificación garantizada):")
print(df_muestra[cat_var].value_counts(normalize=True))
print("\nPrimeros registros procesados:")
print(df_muestra[[cat_var, col_num]].head())

--- Fase 1: Corrupción ---
Nulos inyectados en Amount: 2094

--- Fase 3: Limpieza de Outliers ---
Registros eliminados por Z-Score local: 191

--- Fase final: Muestreo y Suavizado ---
Tamaño de la muestra final: 2752

Distribución de clases (Estratificación garantizada):
Class
0.0    0.996003
1.0    0.003997
Name: proportion, dtype: float64

Primeros registros procesados:
       Class     Amount
8822     0.0  13.501861
3890     0.0  23.882286
7291     0.0  13.501861
9746     0.0  13.501861
13682    0.0  13.501861


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# --- PREPARACIÓN ---
# 'df' es dataset original tras imputar los nulos
df_sin_limpiar = df.copy()

# --- CASO A: CON LIMPIEZA (Paso 3) ---
z_scores = df.groupby(cat_var)[col_num].transform(lambda x: (x - x.mean()) / x.std())
df_limpio = df[z_scores.abs() <= 3].copy()

# --- EVALUACIÓN COMPARATIVA (Punto 5.1) ---

def evaluar(dataset, titulo):
    X = dataset.drop(columns=[cat_var])
    y = dataset[cat_var]
    X_scaled = StandardScaler().fit_transform(X)

    # Modelos
    modelo_lin = LogisticRegression(max_iter=1000).fit(X_scaled, y)
    modelo_arb = DecisionTreeClassifier().fit(X, y)

    print(f"\nResultados para: {titulo}")
    print(f"Precisión Lineal: {modelo_lin.score(X_scaled, y):.4f}")
    print(f"Precisión Árbol: {modelo_arb.score(X, y):.4f}")

# Ejecución
evaluar(df_limpio, "CON paso 3 (Z-Score aplicado)")
evaluar(df_sin_limpiar, "SIN paso 3 (Outliers presentes)")


Resultados para: CON paso 3 (Z-Score aplicado)
Precisión Lineal: 0.9992
Precisión Árbol: 1.0000

Resultados para: SIN paso 3 (Outliers presentes)
Precisión Lineal: 0.9992
Precisión Árbol: 1.0000
